In [106]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Path to the folder containing all CSV files
folder_path = "../data/clean/"

# Combine all CSV files into a single dataframe
all_dataframes = []
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        temp_df = pd.read_csv(file_path)
        all_dataframes.append(temp_df)

# Merge all dataframes
df = pd.concat(all_dataframes, ignore_index=True)

target_cols = ["MOST_POSITIVE_OR_LEAST_NEGATIVE", "NEUTRAL_OR_MIDDLE_CATEGORY", "MOST_NEGATIVE_OR_LEAST_POSITIVE"]
for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Make sure ANSWER1 and ANSWER2 columns exist and convert them to numeric if desired
if "ANSWER1" in df.columns:
    df["ANSWER1"] = pd.to_numeric(df["ANSWER1"], errors='coerce')
if "ANSWER2" in df.columns:
    df["ANSWER2"] = pd.to_numeric(df["ANSWER2"], errors='coerce')

# Update the target columns by replacing 9999 values
df["MOST_POSITIVE_OR_LEAST_NEGATIVE"] = np.where(
    df["MOST_POSITIVE_OR_LEAST_NEGATIVE"] == 9999,
    df["ANSWER1"],
    df["MOST_POSITIVE_OR_LEAST_NEGATIVE"]
)

df["MOST_NEGATIVE_OR_LEAST_POSITIVE"] = np.where(
    df["MOST_NEGATIVE_OR_LEAST_POSITIVE"] == 9999,
    df["ANSWER2"],
    df["MOST_NEGATIVE_OR_LEAST_POSITIVE"]
)

df["NEUTRAL_OR_MIDDLE_CATEGORY"] = np.where(
    df["NEUTRAL_OR_MIDDLE_CATEGORY"] == 9999,
    0,
    df["NEUTRAL_OR_MIDDLE_CATEGORY"]
)

# Selecting relevant columns
features = ["BYCOND", "descrip_E", "SURVEYR", "QUESTION", "INDICATORENG", "SUBINDICATORENG"]
targets = ["MOST_POSITIVE_OR_LEAST_NEGATIVE", "NEUTRAL_OR_MIDDLE_CATEGORY", "MOST_NEGATIVE_OR_LEAST_POSITIVE"]

df = df[features + targets]

# Replace '9999' values in sentiment columns with NaN, then drop them
df.replace(9999, np.nan, inplace=True)
df.dropna(subset=targets, inplace=True)  # Drop rows where any target column had 9999

# Drop remaining NaN values
df.dropna(inplace=True)

# Ensure SURVEYR (year) is numeric
df["SURVEYR"] = pd.to_numeric(df["SURVEYR"], errors="coerce")  # Convert SURVEYR to numeric
df.dropna(subset=["SURVEYR"], inplace=True)  # Drop rows where SURVEYR is NaN after conversion

# Encoding categorical features
label_encoders = {}
categorical_cols = ["BYCOND", "descrip_E", "QUESTION", "INDICATORENG", "SUBINDICATORENG"]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))  # Convert all categorical columns to string before encoding
    label_encoders[col] = le

# Ensure target columns are numeric
for target in targets:
    df[target] = pd.to_numeric(df[target], errors="coerce")
df.dropna(subset=targets, inplace=True)  # Drop rows where targets could not be converted to numeric

# Splitting data
X = df[features]
y = df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Store original test values before transformation
X_test_original = X_test.copy()

# Scaling numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Training XGBoost models
models = {}
for i, sentiment in enumerate(targets):
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train.iloc[:, i])  # Train the model
    models[sentiment] = model

# Predictions
y_pred = np.column_stack([models[sent].predict(X_test) for sent in targets])

# Evaluation
for i, sentiment in enumerate(targets):
    print(f"{sentiment} - MAE: {mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]):.2f}")

# Function to predict sentiment percentages
def predict_sentiment(new_data):
    new_data_df = pd.DataFrame([new_data], columns=features)
    for col in categorical_cols:
        if new_data_df[col][0] not in label_encoders[col].classes_:
            label_encoders[col].classes_ = np.append(label_encoders[col].classes_, new_data_df[col][0])
        new_data_df[col] = label_encoders[col].transform(new_data_df[col].astype(str))
    new_data_df["SURVEYR"] = pd.to_numeric(new_data_df["SURVEYR"], errors="coerce")  # Ensure SURVEYR is numeric
    new_data_df = scaler.transform(new_data_df)
    predictions = {sent: models[sent].predict(new_data_df)[0] for sent in targets}
    return predictions

def predict_sentiment(new_data):
    new_data_df = pd.DataFrame([new_data], columns=features)
    
    for col in categorical_cols:  # Iterate through categorical columns
        # Handle unseen labels
        if new_data_df[col][0] not in label_encoders[col].classes_:
            # Add unseen label to the encoder's classes_
            label_encoders[col].classes_ = np.append(label_encoders[col].classes_, new_data_df[col][0])
        
        # Transform the column using the updated encoder
        try:
            new_data_df[col] = label_encoders[col].transform(new_data_df[col].astype(str))
        except ValueError as e:
            print(f"Warning: Unable to transform column '{col}' with value '{new_data_df[col][0]}'. Setting default value of -1.")
            new_data_df[col] = -1  # Assign a default value for unseen labels
    
    # Ensure SURVEYR is numeric
    new_data_df["SURVEYR"] = pd.to_numeric(new_data_df["SURVEYR"], errors="coerce")  # Handle numeric conversion
    new_data_df.fillna(0, inplace=True)  # Fill any remaining NaN values
    
    # Scale the data
    new_data_df = scaler.transform(new_data_df)
    
    # Predict sentiment percentages
    predictions = {sent: models[sent].predict(new_data_df)[0] for sent in targets}
    return predictions

MOST_POSITIVE_OR_LEAST_NEGATIVE - MAE: 6.27
NEUTRAL_OR_MIDDLE_CATEGORY - MAE: 2.70
MOST_NEGATIVE_OR_LEAST_POSITIVE - MAE: 5.13


In [108]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "",
        "descrip_E": "Health Canada",
        "SURVEYR": "2020",
        "QUESTION":"Q80",
        "TITLE_E": "Question 80. I would describe my workplace as being psychologically healthy.",
        "INDICATORENG": "Workplace well-being","SUBINDICATORENG": "A psychologically healthy workplace"
    },
    {
        "BYCOND": "Q93 = 2",
        "descrip_E": "Working remotely",
        "SURVEYR": "2022",
        "QUESTION":"Q72g",
        "TITLE_E": "Question 72g. Overall, to what extent do the following factors cause you work-related stress? Lack of control or input in decision-making",
        "INDICATORENG": "Workplace well-being","SUBINDICATORENG": "Work-related stress"
    },
    {
        "BYCOND": "D121 = 10",
        "descrip_E": "Southern Asia",
        "SURVEYR": "2022",
        "QUESTION":"Q96b",
        "TITLE_E": "Question 96b. In general, which of the following activities do you feel are best completed at a Government of Canada location? Â… Participate in team building activities",
        "INDICATORENG": "Workplace","SUBINDICATORENG": "Future of work"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 70.11
Neutral or Middle Category: 13.22
Most Negative or Least Positive: 16.97

Sample 2 Prediction:
Most Positive or Least Negative: 60.20
Neutral or Middle Category: 19.28
Most Negative or Least Positive: 18.91

Sample 3 Prediction:
Most Positive or Least Negative: 44.13
Neutral or Middle Category: 0.17
Most Negative or Least Positive: 53.75



In [110]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "Q114 = 7",
        "descrip_E": "British Columbia",
        "SURVEYR": "2019",
        "QUESTION":"Q21",
        "TITLE_E": "Question 21. In my work unit, individuals behave in a respectful manner.",
        "INDICATORENG": "Workplace","SUBINDICATORENG": "Diversity and inclusion"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 84.11
Neutral or Middle Category: 6.86
Most Negative or Least Positive: 7.86



In [112]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "Q114 = 7",
        "descrip_E": "British Columbia",
        "SURVEYR": "2023",
        "QUESTION":"Q21",
        "TITLE_E": "Question 21. In my work unit, individuals behave in a respectful manner.",
        "INDICATORENG": "Workplace","SUBINDICATORENG": "Diversity and inclusion"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 87.31
Neutral or Middle Category: 5.23
Most Negative or Least Positive: 6.72

